<img src="https://github.com/hernancontigiani/ceia_memorias_especializacion/raw/master/Figures/logoFIUBA.jpg" width="500" align="center">

# **Procesamiento de Lenguaje Natural**

## **Desafio, Traductor**


**Este desafío puede realizarse de manera individual o en pareja. En caso de realizarlo en pareja, solo uno de los integrantes deberá subir el desafío.**


Nombre(s):

- Alejandro Valle


### **Consigna**

- Replicar el modelo traductor desarrollado en clase (https://github.com/FIUBA-Posgrado-Inteligencia-Artificial/procesamiento_lenguaje_natural/blob/set_2026/Clase%206/C%C3%B3digo/Traductor.ipynb) y extender su entrenamiento utilizando un conjunto de datos más amplio y secuencias de mayor longitud.
- Modificar valores de hiperparámetros (por ejemplo, el número de unidades en las capas LSTM) y analizar su impacto en el desempeño del traductor.
- Analizar el impacto del número de neuronas en las capas recurrentes, comparando el desempeño de distintas configuraciones del modelo.
- Generar y presentar al menos cinco ejemplos de traducciones producidas por el modelo entrenado.
- Interpretar a detalle los resultados obtenidos, considerando métricas de evaluación, calidad de las traducciones y posibles limitaciones del enfoque utilizado.

### **Actividades opcionales**

- Incorporar embeddings preentrenados para ambos idiomas y evaluar su efecto sobre el rendimiento del modelo.
- Experimentar con diferentes estrategias de generación de secuencias, como muestreo aleatorio (sampling) o búsqueda por haz (beam search).
- Implementar y entrenar una versión equivalente del modelo utilizando PyTorch, comparando los resultados con la implementación original.


## Resolución

Se parte del traductor seq2seq de la clase 6 (inglés → español, encoder y decoder LSTM, GloVe en el encoder) y se lo extiende:

1. **Más datos y secuencias más largas**: 50.000 pares de oraciones (5 veces los 10.000 de clase) y longitudes máximas que cubren la oración más larga del conjunto, en lugar de truncar a 20/22 palabras.
2. **Barrido del número de neuronas** de las LSTM (64, 128, 256 y 512; en clase 256), con todo lo demás fijo.
3. **Evaluación**: pérdida y accuracy **sin contar el padding**, BLEU sobre un conjunto de test con todas las traducciones de referencia del corpus, BLEU según la longitud de la oración y ejemplos de traducción.
4. **Opcional**: búsqueda por haz (beam search) y muestreo aleatorio con temperatura, sobre el mejor modelo.

### Cambios respecto del notebook de clase

| | Clase | Este notebook | Motivo |
|---|---|---|---|
| Pares de oraciones | 10.000 | 50.000 | Consigna: conjunto de datos más amplio |
| Longitud máxima (EN / ES) | 20 / 22, truncando | La de la oración más larga, sin truncar | Consigna: secuencias más largas. Además, `pad_sequences` trunca por defecto **al inicio**, y en el decoder eso borraba el `<sos>` |
| Vocabulario máximo | 8.000 | 10.000 | Con 50.000 pares el español tiene ~18.000 palabras distintas |
| Ajuste del tokenizer en español | `["<sos>", "<eos>"] + output_sentences` | `output_sentences_inputs + output_sentences` | `<sos>` aparecía una sola vez: con más datos su índice supera el tamaño del vocabulario y se pierde (bug) |
| `nb_words` del encoder | `min(V, len(word_index))` | `min(V, len(word_index) + 1)` | Los índices empiezan en 1: la última palabra quedaba fuera de la matriz de embeddings (bug) |
| Contracciones en inglés | `don't`, `I'm` | `do n't`, `I 'm` | GloVe separa así las contracciones; sin esto quedan con vector nulo y congelado |
| `¡` | Quedaba pegado a la palabra (`¡hola`) | Se elimina, como `¿` | Evita duplicar palabras en el vocabulario |
| Targets | One-hot generado con un generador de Python | Índices enteros + entropía cruzada *sparse* | Es el mismo cálculo, con mucha menos memoria y tiempo |
| Pérdida y accuracy | Cuentan el padding | Ignoran el padding | La mayoría de las posiciones del decoder son relleno: inflaban la accuracy |
| Validación | Último 15% de los pares | Train / val / test separados por oración en inglés | Una misma oración en inglés con varias traducciones no puede quedar en train y en test |
| Batch / learning rate | 64 / 5e-4 | 128 / 1e-3 | Con 5 veces más datos; iguales para todas las configuraciones |
| Inferencia | `predict` palabra por palabra, oración por oración | El decoder se llama sobre muchas oraciones a la vez | Necesario para calcular BLEU sobre miles de oraciones |

La arquitectura, el dropout, el optimizador (Adam), los callbacks (`ReduceLROnPlateau`, `EarlyStopping`) y la inferencia con modelos separados de encoder y decoder son los de clase.

**Cómo correrlo**: en Google Colab con GPU (T4), *Ejecutar todo*. Si el entorno no tiene GPU, la celda **GPU** detiene el notebook con un aviso, porque entrenar en CPU tarda horas. Los pesos y las curvas de cada configuración se guardan en Google Drive (`MyDrive/nlp1_models/desafio4`; pide autorización al comienzo). Si la sesión se corta, al volver a ejecutar se cargan las configuraciones ya entrenadas y sólo se entrenan las que faltan.

In [ ]:
import json
import os
import pickle
import re
import sys
import time
import urllib.request
import warnings
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
from nltk.translate.bleu_score import corpus_bleu
from tensorflow import keras
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.layers import LSTM, Dense, Dropout, Embedding, Input
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.preprocessing.text import Tokenizer, text_to_word_sequence

# el dataset ya se mezcla en tf.data: este aviso de Keras no aplica
warnings.filterwarnings("ignore", message=".*`shuffle=True` was passed.*")
# nltk avisa cuando un grupo de oraciones no tiene ningún 4-grama en común (BLEU = 0)
warnings.filterwarnings("ignore", module="nltk")
# al recargar pesos guardados el optimizador no se usa: este aviso no aplica
warnings.filterwarnings("ignore", message="Skipping variable loading for optimizer")
pd.set_option("display.max_colwidth", None)

print(f"TensorFlow {tf.__version__} | Keras {keras.__version__}")

### GPU

Entrenar las cuatro configuraciones en CPU tarda horas. Si el entorno de Colab no tiene GPU, el notebook se detiene acá con un aviso.

In [ ]:
# para correrlo igual en CPU (horas), poner REQUIRE_GPU = False
REQUIRE_GPU = True

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    details = tf.config.experimental.get_device_details(gpus[0])
    print(f"GPU: {details.get('device_name', gpus[0].name)}")
elif REQUIRE_GPU:
    raise RuntimeError(
        "No hay GPU. En Colab: Entorno de ejecución > Cambiar tipo de entorno de ejecución > "
        "GPU T4, y volver a ejecutar todo. Para entrenar igual en CPU, poner REQUIRE_GPU = False."
    )
else:
    print("No hay GPU: se entrena en CPU (REQUIRE_GPU = False)")

### Configuración

Todos los hiperparámetros del experimento están en esta celda. Para el barrido de neuronas sólo cambia `n_units`: los datos, el vocabulario, el batch y el learning rate son los mismos para todas las configuraciones, así las diferencias se deben al tamaño de las LSTM.

In [ ]:
SEED = 40  # misma semilla que en clase para mezclar el corpus

NUM_SENTENCES = 50_000  # clase: 10.000 pares
MAX_VOCAB_SIZE = 10_000  # clase: 8.000 (ver la cobertura del vocabulario más abajo)
EMBED_DIM = 50  # GloVe de 50 dimensiones, como en clase
DROPOUT = 0.3  # como en clase

UNITS_LIST = [64, 128, 256, 512]  # neuronas de las LSTM a comparar (clase: 256)
BATCH_SIZE = 128  # clase: 64
LEARNING_RATE = 1e-3  # clase: 5e-4
EPOCHS = 30  # tope, como en clase: EarlyStopping corta antes

VAL_FRACTION = 0.10  # fracción de las oraciones en inglés distintas
TEST_FRACTION = 0.05
N_STRATEGY_EVAL = 500  # oraciones de test para comparar estrategias de decodificación

CLASS_MAX_INPUT_LEN, CLASS_MAX_OUT_LEN = 20, 22  # longitudes fijas del notebook de clase

Los pesos y el historial de cada configuración se guardan en `MODELS_DIR`. Bajo Colab es una carpeta de Google Drive, para no perderlos si la sesión se desconecta: la celda pide autorización al comienzo de la corrida.

In [ ]:
# En Google Colab los archivos de la sesión se pierden al desconectarse: se guardan en Drive
if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive")
    MODELS_DIR = Path("/content/drive/MyDrive/nlp1_models/desafio4")
else:
    MODELS_DIR = Path("models_desafio4")
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print(f"Los pesos y las curvas se guardan en: {MODELS_DIR.resolve()}")

### Datos

Se usa el mismo corpus que en clase: pares inglés-español de [Tatoeba](https://tatoeba.org) publicados por TensorFlow (`spa-eng`, ~119.000 pares). Son oraciones cortas y coloquiales, ordenadas de la más corta a la más larga en el archivo, por eso se mezclan antes de tomar el subconjunto.

In [ ]:
DATA_URL = "http://storage.googleapis.com/download.tensorflow.org/data/spa-eng.zip"
DATA_PATH = Path("spa-eng/spa.txt")

if not DATA_PATH.exists():
    urllib.request.urlretrieve(DATA_URL, "spa-eng.zip")
    with zipfile.ZipFile("spa-eng.zip") as zf:
        zf.extractall(".")

with open(DATA_PATH, encoding="utf-8") as f:
    lines = f.read().split("\n")[:-1]

# mismo mezclado que en clase
np.random.seed(SEED)
np.random.shuffle(lines)

all_pairs = [line.rstrip().split("\t")[:2] for line in lines if "\t" in line]
print(f"Pares en el corpus completo: {len(all_pairs):,}")

In [ ]:
# Tatoeba repite oraciones en inglés con varias traducciones válidas ("Go." -> "Ve.", "Vete.",
# "Vaya."). Guardamos todas las traducciones de cada oración: son las referencias del BLEU
all_translations = defaultdict(list)
for en, es in all_pairs:
    all_translations[en].append(es)

pairs = all_pairs[:NUM_SENTENCES]
input_sentences = [en for en, _ in pairs]
target_sentences = [es for _, es in pairs]
output_sentences = [es + " <eos>" for es in target_sentences]
output_sentences_inputs = ["<sos> " + es for es in target_sentences]

unique_inputs = set(input_sentences)
n_multi = sum(len(all_translations[en]) > 1 for en in unique_inputs)
print(f"Pares usados: {len(pairs):,} (clase: 10.000)")
print(f"Oraciones en inglés distintas: {len(unique_inputs):,}")
print(f"... con más de una traducción en el corpus: {n_multi:,}\n")

print(input_sentences[0])
print(output_sentences[0])
print(output_sentences_inputs[0])

### Tokenización

Como en clase, cada idioma tiene su `Tokenizer` de Keras (minúsculas, sin puntuación, las `MAX_VOCAB_SIZE` palabras más frecuentes). Hay tres cambios:

- **Contracciones en inglés.** El vocabulario de GloVe separa las contracciones como `do n't`, `i 'm`, `tom 's`. El tokenizer de Keras, en cambio, deja `don't` o `i'm` como una sola palabra, que no existe en GloVe: su embedding queda en cero y, como la capa está congelada, el modelo no puede distinguir `don't` de `can't` o `i'm`. Separamos las contracciones antes de tokenizar.
- **`<sos>` dentro del vocabulario.** En clase el tokenizer del español se ajustaba con `["<sos>", "<eos>"] + output_sentences`: `<sos>` aparecía **una sola vez** y quedaba entre las palabras menos frecuentes. Con 10.000 pares su índice era 3.434 (< 8.000) y no pasaba nada; con 50.000 pares supera el tamaño del vocabulario, `texts_to_sequences` lo descarta de las entradas del decoder y la inferencia busca un índice que no existe en el embedding. Ajustando con las entradas y las salidas del decoder, `<sos>` y `<eos>` aparecen una vez por oración.
- **`nb_words`.** Los índices del tokenizer empiezan en 1 (el 0 es el padding), así que el `Embedding` necesita `len(word_index) + 1` filas. En clase faltaba el `+ 1` para el inglés.

In [ ]:
def split_clitics(text):
    """Separa las contracciones del inglés como el vocabulario de GloVe:
    "don't" -> "do n't", "can't" -> "ca n't", "I'm" -> "I 'm", "Tom's" -> "Tom 's"."""
    text = text.replace("’", "'")
    text = re.sub(r"n't\b", " n't", text, flags=re.IGNORECASE)
    return re.sub(r"'(s|m|re|ve|ll|d)\b", r" '\1", text, flags=re.IGNORECASE)


print(split_clitics("I'm sure Tom's dog can't swim, but I don't know why."))

input_texts = [split_clitics(s) for s in input_sentences]
input_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE)
input_tokenizer.fit_on_texts(input_texts)
input_integer_seq = input_tokenizer.texts_to_sequences(input_texts)
word2idx_inputs = input_tokenizer.word_index

# como en clase se eliminan los signos de puntuación sin tocar < y > (de <sos> y <eos>);
# se agrega ¡ para que "¡hola" y "hola" sean la misma palabra
OUTPUT_FILTERS = '!"#$%&()*+,-./:;=¿?¡@[\\]^_`{|}~\t\n'
output_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters=OUTPUT_FILTERS)
output_tokenizer.fit_on_texts(output_sentences_inputs + output_sentences)
output_integer_seq = output_tokenizer.texts_to_sequences(output_sentences)
output_input_integer_seq = output_tokenizer.texts_to_sequences(output_sentences_inputs)
word2idx_outputs = output_tokenizer.word_index

# índices válidos: de 0 (padding) a num_words - 1
nb_words = min(MAX_VOCAB_SIZE, len(word2idx_inputs) + 1)
num_words_output = min(MAX_VOCAB_SIZE, len(word2idx_outputs) + 1)
sos_idx, eos_idx = word2idx_outputs["<sos>"], word2idx_outputs["<eos>"]
assert max(sos_idx, eos_idx) < num_words_output

# así quedaba <sos> con el ajuste de clase
class_tokenizer = Tokenizer(num_words=MAX_VOCAB_SIZE, filters=OUTPUT_FILTERS)
class_tokenizer.fit_on_texts(["<sos>", "<eos>"] + output_sentences)
print(
    f"\nCon el ajuste de clase, <sos> tendría el índice {class_tokenizer.word_index['<sos>']:,} "
    f"(vocabulario máximo: {MAX_VOCAB_SIZE:,})"
)
print(f"Ahora: <sos> = {sos_idx}, <eos> = {eos_idx}\n")


def token_coverage(tokenizer, num_words, skip=("<sos>", "<eos>")):
    """Fracción de las apariciones de palabras que quedan dentro del vocabulario."""
    counts = {w: c for w, c in tokenizer.word_counts.items() if w not in skip}
    kept = sum(c for w, c in counts.items() if tokenizer.word_index[w] < num_words)
    return kept / sum(counts.values())


pd.DataFrame(
    {
        "palabras distintas": [len(word2idx_inputs), len(word2idx_outputs) - 2],
        "tamaño del vocabulario (con el 0)": [nb_words, num_words_output],
        "% de apariciones dentro del vocabulario": [
            round(100 * token_coverage(input_tokenizer, nb_words), 2),
            round(100 * token_coverage(output_tokenizer, num_words_output), 2),
        ],
    },
    index=["inglés (encoder)", "español (decoder)"],
)

Las palabras fuera del vocabulario **no se reemplazan por un token especial: `texts_to_sequences` las elimina** (igual que en clase). En español esto afecta sobre todo a nombres propios y conjugaciones poco frecuentes: el modelo nunca podrá generarlas.

### Longitud de las secuencias

In [ ]:
input_lens = np.array([len(s) for s in input_integer_seq])
output_lens = np.array([len(s) for s in output_integer_seq])  # palabras + <eos>
assert (output_lens == np.array([len(s) for s in output_input_integer_seq])).all()

length_stats = (
    pd.DataFrame(
        {
            "inglés (encoder)": pd.Series(input_lens).describe(percentiles=[0.5, 0.9, 0.99, 0.999]),
            "español + <eos> (decoder)": pd.Series(output_lens).describe(
                percentiles=[0.5, 0.9, 0.99, 0.999]
            ),
        }
    )
    .T[["mean", "50%", "90%", "99%", "99.9%", "max"]]
    .round(1)
)
length_stats.columns = ["media", "mediana", "p90", "p99", "p99.9", "máx"]

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, lens, limit, title in [
    (axes[0], input_lens, CLASS_MAX_INPUT_LEN, "Inglés (entrada del encoder)"),
    (axes[1], output_lens, CLASS_MAX_OUT_LEN, "Español + <eos> (decoder)"),
]:
    ax.hist(lens, bins=range(1, lens.max() + 2), color="tab:blue")
    ax.axvline(limit + 0.5, color="tab:red", linestyle="--", label=f"límite de clase ({limit})")
    ax.set_yscale("log")
    ax.set_xlabel("palabras")
    ax.set_ylabel("oraciones (escala log)")
    ax.set_title(title)
    ax.legend()
plt.tight_layout()
plt.show()

length_stats

Las oraciones de Tatoeba son cortas (mediana de ~6 palabras), pero la cola llega mucho más lejos que los límites de clase. Con el `pad_sequences` de clase esas oraciones se truncaban **al inicio** (`truncating="pre"` es el valor por defecto): en el encoder se perdía el comienzo de la oración y en el decoder se perdía el `<sos>`, de modo que el modelo aprendía con un ejemplo mal armado.

Se toman como longitudes máximas las de la oración más larga del conjunto: no se trunca nada. El costo es más padding, que no afecta al aprendizaje porque la pérdida lo ignora (ver más abajo).

In [ ]:
max_input_len = int(input_lens.max())
max_out_len = int(output_lens.max())

# como en clase: padding al inicio en el encoder (así el último estado de la LSTM corresponde
# a la última palabra) y al final en el decoder
encoder_input_sequences = pad_sequences(input_integer_seq, maxlen=max_input_len)
decoder_input_sequences = pad_sequences(
    output_input_integer_seq, maxlen=max_out_len, padding="post"
)
decoder_output_sequences = pad_sequences(output_integer_seq, maxlen=max_out_len, padding="post")

assert (decoder_input_sequences[:, 0] == sos_idx).all()
assert encoder_input_sequences.max() < nb_words
assert decoder_output_sequences.max() < num_words_output

# qué hacía el truncado de clase con estos datos
class_dec_in = pad_sequences(output_input_integer_seq, maxlen=CLASS_MAX_OUT_LEN, padding="post")
n_truncated = int(np.sum((input_lens > CLASS_MAX_INPUT_LEN) | (output_lens > CLASS_MAX_OUT_LEN)))
n_lost_sos = int(np.sum(class_dec_in[:, 0] != sos_idx))
print(
    f"max_input_len = {max_input_len} (clase: {CLASS_MAX_INPUT_LEN}) | "
    f"max_out_len = {max_out_len} (clase: {CLASS_MAX_OUT_LEN})"
)
print(f"Pares que la clase hubiera truncado: {n_truncated} ({n_lost_sos} perdían el <sos>)")

pad_share = 1 - (decoder_output_sequences > 0).mean()
print(f"Posiciones de salida del decoder que son padding: {pad_share:.1%}")
print(
    f"Con la accuracy de clase (que cuenta el padding), un modelo que sólo predijera padding "
    f"acertaría el {pad_share:.0%} sin traducir nada"
)

### División en entrenamiento, validación y test

Una misma oración en inglés aparece varias veces con traducciones distintas. Si se dividieran los **pares** al azar, la oración podría estar en entrenamiento y en test a la vez, y el test mediría memoria en lugar de traducción. Por eso se dividen las **oraciones en inglés distintas** (85% / 10% / 5%) y cada par va al conjunto de su oración.

- **Entrenamiento y validación** se usan como en clase (validación para `EarlyStopping` y `ReduceLROnPlateau`).
- **Test** sólo se usa al final: se traduce una vez cada oración en inglés y se la compara con **todas** sus traducciones del corpus completo.

El dataset de `tf.data` se arma con los índices enteros (sin one-hot, que con un vocabulario de 10.000 palabras ocuparía `batch × max_out_len × 10.000` números por batch) y se mezcla en cada epoch.

In [ ]:
unique_en = sorted(unique_inputs)
rng = np.random.default_rng(SEED)
rng.shuffle(unique_en)
n_test = int(len(unique_en) * TEST_FRACTION)
n_val = int(len(unique_en) * VAL_FRACTION)
split_of = {en: "test" for en in unique_en[:n_test]}
split_of.update({en: "val" for en in unique_en[n_test : n_test + n_val]})
split = np.array([split_of.get(en, "train") for en in input_sentences])


def make_dataset(mask, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices(
        (
            (encoder_input_sequences[mask], decoder_input_sequences[mask]),
            decoder_output_sequences[mask],
        )
    )
    if shuffle:
        ds = ds.shuffle(int(mask.sum()), seed=SEED)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(split == "train", shuffle=True)
val_ds = make_dataset(split == "val")

# test: la primera aparición de cada oración en inglés
first_occurrence = {}
for i, en in enumerate(input_sentences):
    if split[i] == "test":
        first_occurrence.setdefault(en, i)
test_idx = np.array(sorted(first_occurrence.values()))

print(f"Pares de entrenamiento: {np.sum(split == 'train'):,}")
print(f"Pares de validación:    {np.sum(split == 'val'):,}")
print(f"Oraciones de test:      {len(test_idx):,} (inglés distintas)")

### Embeddings GloVe para el encoder

Igual que en clase: vectores GloVe de 50 dimensiones para las palabras en inglés, congelados (`trainable=False`). El decoder aprende sus propios embeddings del español.

In [ ]:
def _is_valid_pickle(path):
    try:
        with open(path, "rb") as f:
            head = f.read(20)
        return b"<html" not in head.lower() and b"<!doctype" not in head.lower()
    except Exception:
        return False


_PKL_PATH = "gloveembedding.pkl"
_FILE_ID = "1KY6avD5I1eI2dxQzMkR3WExwKwRq2g94"

if not os.path.exists(_PKL_PATH) or not _is_valid_pickle(_PKL_PATH):
    print("Descargando gloveembedding.pkl desde Google Drive...")
    if os.path.exists(_PKL_PATH):
        os.remove(_PKL_PATH)
    try:
        import gdown

        gdown.download(id=_FILE_ID, output=_PKL_PATH, quiet=False)
    except Exception:
        os.system(
            f"curl -L -o {_PKL_PATH} "
            f'"https://drive.google.com/u/0/uc?id={_FILE_ID}&export=download&confirm=t"'
        )
    if not _is_valid_pickle(_PKL_PATH):
        raise ValueError("El archivo descargado no es un pickle válido.")
    print("Descarga completada.")
else:
    print("gloveembedding.pkl ya disponible.")

In [ ]:
def load_glove_embeddings(pkl_path):
    """Carga el pickle de GloVe y devuelve (embeddings, word2idx)."""
    max_bytes = 2**28 - 1
    raw = bytearray()
    sz = os.path.getsize(pkl_path)
    with open(pkl_path, "rb") as f:
        for _ in range(0, sz, max_bytes):
            raw += f.read(max_bytes)
    embeddings = pickle.loads(raw)
    idx_array = np.arange(embeddings.shape[0])
    word2idx = dict(zip(embeddings["word"], idx_array))
    return embeddings, word2idx


def get_word_embedding(word, embeddings, word2idx, n_features=50):
    i = word2idx.get(word, -1)
    return embeddings[i]["embedding"] if i != -1 else np.zeros(n_features)


def build_embedding_matrix(word2idx_inputs, embeddings, word2idx_glove, nb_words, embed_dim=50):
    matrix = np.zeros((nb_words, embed_dim))
    for word, i in word2idx_inputs.items():
        if i < nb_words:
            vec = get_word_embedding(word, embeddings, word2idx_glove, embed_dim)
            if vec is not None and len(vec) > 0:
                matrix[i] = vec
    return matrix


glove_embeddings, glove_word2idx = load_glove_embeddings(_PKL_PATH)
embedding_matrix = build_embedding_matrix(
    word2idx_inputs, glove_embeddings, glove_word2idx, nb_words, EMBED_DIM
)
print(f"Embeddings nulos: {np.sum(np.sum(embedding_matrix**2, axis=1) == 0)} de {nb_words:,} filas")

# ¿cuántas apariciones de palabras quedan sin vector, con y sin separar las contracciones?
tokens_raw = [w for s in input_sentences for w in text_to_word_sequence(s)]
tokens_split = [w for s in input_texts for w in text_to_word_sequence(s)]
missing_raw = Counter(w for w in tokens_raw if w not in glove_word2idx)
missing_split = Counter(w for w in tokens_split if w not in glove_word2idx)
glove_missing_raw = sum(missing_raw.values()) / len(tokens_raw)
glove_missing_split = sum(missing_split.values()) / len(tokens_split)
print(
    f"Apariciones sin vector GloVe: {glove_missing_raw:.1%} sin separar "
    f"las contracciones -> {glove_missing_split:.1%} separándolas"
)
print(f"Más frecuentes sin vector (sin separar): {missing_raw.most_common(8)}")
print(f"Más frecuentes sin vector (separando):   {missing_split.most_common(8)}")

### Modelo seq2seq

La arquitectura es la de clase:

- **Encoder**: embedding GloVe congelado → dropout → LSTM. Sólo se conservan sus estados finales `(h, c)`, que resumen toda la oración en inglés en un vector de `n_units` números (el **vector de contexto**).
- **Decoder**: embedding propio del español (de dimensión `n_units`, como en clase) → dropout → LSTM que arranca desde los estados del encoder → `Dense` con softmax sobre el vocabulario español.
- **Entrenamiento con *teacher forcing***: el decoder recibe la traducción correcta desplazada un lugar (`<sos> w1 w2 ...`) y debe predecir la siguiente palabra (`w1 w2 ... <eos>`).

Para inferencia se arman dos modelos que **reutilizan las mismas capas** (y por lo tanto los pesos entrenados): el encoder, que devuelve `(h, c)`, y un decoder que avanza una palabra por vez.

**Pérdida y accuracy sin padding.** En clase la pérdida y la accuracy se promediaban sobre todas las posiciones del decoder, incluido el relleno. Como la mayoría son padding (ver arriba), predecir "0" en esas posiciones es trivial e infla la accuracy. Aquí la entropía cruzada y la accuracy se promedian sólo sobre las palabras reales (incluido `<eos>`). Con esta pérdida el modelo nunca aprende a predecir el padding, así que la accuracy de clase ya no tendría sentido: no se registra. Con esta pérdida, `exp(val_loss)` es la **perplejidad** por palabra.

In [ ]:
def masked_loss(y_true, y_pred):
    """Entropía cruzada promedio sobre las posiciones que no son padding."""
    mask = tf.cast(tf.not_equal(y_true, 0), y_pred.dtype)
    loss = keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    return tf.reduce_sum(loss * mask) / tf.reduce_sum(mask)


def masked_accuracy(y_true, y_pred):
    """Fracción de palabras acertadas, sin contar las posiciones de padding."""
    y_true = tf.cast(y_true, tf.int64)
    mask = tf.not_equal(y_true, 0)
    hits = tf.logical_and(tf.equal(tf.argmax(y_pred, axis=-1), y_true), mask)
    return tf.reduce_sum(tf.cast(hits, tf.float32)) / tf.reduce_sum(tf.cast(mask, tf.float32))


def build_seq2seq(n_units):
    """Devuelve (modelo de entrenamiento, encoder de inferencia, decoder de inferencia)."""
    # --- encoder ---
    enc_inputs = Input(shape=(max_input_len,), name="encoder_inputs")
    enc_emb_layer = Embedding(
        input_dim=nb_words,
        output_dim=EMBED_DIM,
        weights=[embedding_matrix],
        trainable=False,
        name="encoder_embedding",
    )
    enc_emb = Dropout(DROPOUT, name="encoder_dropout")(enc_emb_layer(enc_inputs))
    enc_lstm_layer = LSTM(n_units, return_state=True, name="encoder_lstm")
    _, state_h, state_c = enc_lstm_layer(enc_emb)

    # --- decoder ---
    dec_inputs = Input(shape=(max_out_len,), name="decoder_inputs")
    dec_emb_layer = Embedding(
        input_dim=num_words_output, output_dim=n_units, name="decoder_embedding"
    )
    dec_emb = Dropout(DROPOUT, name="decoder_dropout")(dec_emb_layer(dec_inputs))
    dec_lstm_layer = LSTM(n_units, return_sequences=True, return_state=True, name="decoder_lstm")
    dec_out, _, _ = dec_lstm_layer(dec_emb, initial_state=[state_h, state_c])
    dec_dense_layer = Dense(num_words_output, activation="softmax", name="decoder_dense")
    model = Model([enc_inputs, dec_inputs], dec_dense_layer(dec_out), name=f"seq2seq_{n_units}")

    # --- inferencia: mismas capas, sin dropout ---
    _, inf_h, inf_c = enc_lstm_layer(enc_emb_layer(enc_inputs))
    encoder_model = Model(enc_inputs, [inf_h, inf_c], name=f"encoder_{n_units}")

    dec_input_single = Input(shape=(1,), name="dec_input_single")
    dec_state_h_in = Input(shape=(n_units,), name="dec_state_h")
    dec_state_c_in = Input(shape=(n_units,), name="dec_state_c")
    single_out, h_out, c_out = dec_lstm_layer(
        dec_emb_layer(dec_input_single), initial_state=[dec_state_h_in, dec_state_c_in]
    )
    decoder_model = Model(
        [dec_input_single, dec_state_h_in, dec_state_c_in],
        [dec_dense_layer(single_out), h_out, c_out],
        name=f"decoder_{n_units}",
    )
    return model, encoder_model, decoder_model


def count_trainable(model):
    return int(sum(np.prod(w.shape) for w in model.trainable_weights))


# la configuración de clase (256 neuronas)
build_seq2seq(256)[0].summary()

In [ ]:
# cómo crece el modelo con el número de neuronas (el embedding GloVe está congelado)
param_rows = []
for n_units in UNITS_LIST:
    m = build_seq2seq(n_units)[0]
    param_rows.append(
        {
            "neuronas LSTM": n_units,
            "parámetros entrenables": count_trainable(m),
            "LSTM (encoder + decoder)": m.get_layer("encoder_lstm").count_params()
            + m.get_layer("decoder_lstm").count_params(),
            "embedding + Dense del decoder": m.get_layer("decoder_embedding").count_params()
            + m.get_layer("decoder_dense").count_params(),
        }
    )
pd.DataFrame(param_rows).set_index("neuronas LSTM")

### Entrenamiento

Se entrena una configuración por cada valor de `UNITS_LIST`, con los mismos callbacks que en clase (`ReduceLROnPlateau` divide el learning rate a la mitad si la pérdida de validación no mejora en 2 epochs; `EarlyStopping` corta tras 4 epochs sin mejora). `ModelCheckpoint` guarda los pesos de la **mejor** epoch y al final se cargan esos.

Cada configuración deja en `MODELS_DIR` sus pesos (`lstm_<n>.weights.h5`) y su historial (`lstm_<n>_history.json`). Si ya existen, se cargan en lugar de reentrenar (se verifica que se hayan entrenado con los mismos datos y vocabulario).

In [ ]:
# si cambia algo de esto, los pesos guardados no sirven
RUN_CONFIG = {
    "num_sentences": NUM_SENTENCES,
    "seed": SEED,
    "nb_words": int(nb_words),
    "num_words_output": int(num_words_output),
    "max_input_len": max_input_len,
    "max_out_len": max_out_len,
    "embed_dim": EMBED_DIM,
    "batch_size": BATCH_SIZE,
    "learning_rate": LEARNING_RATE,
}


def train_or_load(n_units):
    name = f"lstm_{n_units}"
    weights_path = MODELS_DIR / f"{name}.weights.h5"
    record_path = MODELS_DIR / f"{name}_history.json"

    keras.utils.set_random_seed(SEED)
    model, encoder_model, decoder_model = build_seq2seq(n_units)
    model.compile(
        loss=masked_loss,
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        metrics=[masked_accuracy],
    )

    if weights_path.exists() and record_path.exists():
        record = json.loads(record_path.read_text())
        assert record["config"] == RUN_CONFIG, (
            f"{record_path} se entrenó con otros datos o vocabulario: "
            "borrar sus archivos y reentrenar"
        )
        model.load_weights(weights_path)
        print(
            f"{name}: se cargan los pesos guardados ({record['epochs_run']} epochs, sin reentrenar)"
        )
    else:
        print(f"Entrenando {name}...")
        callbacks = [
            ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5, verbose=1),
            EarlyStopping(monitor="val_loss", patience=4, verbose=1),
            ModelCheckpoint(
                str(weights_path), monitor="val_loss", save_best_only=True, save_weights_only=True
            ),
        ]
        start = time.time()
        hist = model.fit(
            train_ds, validation_data=val_ds, epochs=EPOCHS, callbacks=callbacks, verbose=2
        )
        seconds = time.time() - start
        model.load_weights(weights_path)  # nos quedamos con la mejor epoch
        record = {
            "n_units": n_units,
            "trainable_params": count_trainable(model),
            "epochs_run": len(hist.history["loss"]),
            "seconds": seconds,
            "history": {k: [float(v) for v in vals] for k, vals in hist.history.items()},
            "config": RUN_CONFIG,
        }
        record_path.write_text(json.dumps(record))
    return {"model": model, "encoder": encoder_model, "decoder": decoder_model, "record": record}

In [ ]:
runs = {}
for n_units in UNITS_LIST:
    runs[n_units] = train_or_load(n_units)

### Curvas de entrenamiento

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for (n_units, run), color in zip(runs.items(), plt.cm.viridis(np.linspace(0, 0.9, len(runs)))):
    hist = run["record"]["history"]
    epochs = range(1, len(hist["loss"]) + 1)
    axes[0].plot(epochs, hist["loss"], "--", color=color, alpha=0.6)
    axes[0].plot(epochs, hist["val_loss"], color=color, label=f"{n_units} neuronas")
    axes[1].plot(epochs, hist["masked_accuracy"], "--", color=color, alpha=0.6)
    axes[1].plot(epochs, hist["val_masked_accuracy"], color=color, label=f"{n_units} neuronas")

axes[0].set_title("Pérdida sin padding (- - train, — val)")
axes[1].set_title("Accuracy sin padding (- - train, — val)")
for ax in axes:
    ax.set_xlabel("epoch")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### Inferencia y evaluación con BLEU

**Decodificación.** En inferencia el decoder no tiene la traducción correcta: empieza con `<sos>` y en cada paso recibe la palabra que él mismo generó. La estrategia de clase es **greedy** (en cada paso, la palabra más probable). Aquí el decoder se llama sobre muchas oraciones a la vez (un batch), en lugar de una llamada a `predict` por palabra y por oración, para poder traducir todo el conjunto de test. Las mismas funciones sirven para el muestreo aleatorio (más abajo).

**BLEU** compara cada traducción con sus referencias contando cuántos n-gramas (1 a 4 palabras seguidas) de la traducción aparecen en alguna referencia, y penaliza las traducciones más cortas que las referencias. Se calcula a nivel corpus, entre 0 y 100. Como referencias se usan **todas** las traducciones de la oración en el corpus completo, en minúsculas y sin puntuación (igual que lo que genera el modelo).

In [ ]:
idx2word_target = {v: k for k, v in word2idx_outputs.items()}


def encode_english(texts):
    """Oraciones en inglés -> secuencias de entrada del encoder (como en entrenamiento)."""
    seqs = input_tokenizer.texts_to_sequences([split_clitics(t) for t in texts])
    return pad_sequences(seqs, maxlen=max_input_len)


def es_tokens(text):
    return text_to_word_sequence(text, filters=OUTPUT_FILTERS)


def ids_to_words(ids):
    words = []
    for i in ids:
        if i == eos_idx:
            break
        words.append(idx2word_target[int(i)])
    return words


def decoder_step(decoder_model, tokens, h, c):
    """Un paso del decoder para un batch: log-probabilidades de la próxima palabra y estados."""
    # Keras no acepta mezclar arrays de numpy y tensores en la misma llamada
    probs, h, c = decoder_model([np.asarray(x) for x in (tokens, h, c)], training=False)
    log_probs = np.log(np.asarray(probs)[:, 0, :] + 1e-12)
    log_probs[:, [0, sos_idx]] = -np.inf  # el padding y <sos> no son salidas válidas
    return log_probs, np.asarray(h), np.asarray(c)


def decode_batch(run, enc_seqs, temperature=None, seed=SEED, batch_size=512):
    """Traduce muchas oraciones a la vez.
    temperature=None: greedy (la palabra más probable en cada paso).
    temperature=T: muestreo aleatorio de la distribución con temperatura T."""
    rng = np.random.default_rng(seed)
    results = []
    for start in range(0, len(enc_seqs), batch_size):
        batch = enc_seqs[start : start + batch_size]
        h, c = run["encoder"](batch, training=False)
        tokens = np.full((len(batch), 1), sos_idx, dtype=np.int32)
        generated = np.full((len(batch), max_out_len), eos_idx, dtype=np.int64)
        finished = np.zeros(len(batch), dtype=bool)
        for t in range(max_out_len):
            log_probs, h, c = decoder_step(run["decoder"], tokens, h, c)
            if temperature is None:
                next_ids = log_probs.argmax(axis=-1)
            else:
                logits = log_probs / temperature
                p = np.exp(logits - logits.max(axis=-1, keepdims=True))
                p /= p.sum(axis=-1, keepdims=True)
                next_ids = (p.cumsum(axis=-1) < rng.random((len(p), 1))).sum(axis=-1)
                next_ids = np.minimum(next_ids, p.shape[1] - 1)
            generated[:, t] = np.where(finished, eos_idx, next_ids)
            finished |= next_ids == eos_idx
            if finished.all():
                break
            tokens = next_ids[:, None].astype(np.int32)
        results.extend(ids_to_words(row) for row in generated)
    return results


def bleu(hypotheses, references):
    if not hypotheses:
        return float("nan")
    return 100 * corpus_bleu(references, hypotheses)


# conjunto de test: entradas, referencias y longitud de la oración en inglés
test_enc = encoder_input_sequences[test_idx]
test_refs = [[es_tokens(es) for es in all_translations[input_sentences[i]]] for i in test_idx]
test_en_len = np.array([len(text_to_word_sequence(input_sentences[i])) for i in test_idx])
print(
    f"Oraciones de test: {len(test_idx):,} | referencias por oración: "
    f"{np.mean([len(r) for r in test_refs]):.2f} en promedio"
)

In [ ]:
test_predictions = {}
summary_rows = []
for n_units, run in runs.items():
    start = time.time()
    test_predictions[n_units] = decode_batch(run, test_enc)
    decode_seconds = time.time() - start

    rec = run["record"]
    hist = rec["history"]
    best = int(np.argmin(hist["val_loss"]))
    summary_rows.append(
        {
            "neuronas LSTM": n_units,
            "parámetros entrenables": rec["trainable_params"],
            "mejor epoch / corridas": f"{best + 1} / {rec['epochs_run']}",
            "val loss": hist["val_loss"][best],
            "val perplejidad": np.exp(hist["val_loss"][best]),
            "train acc. sin padding": hist["masked_accuracy"][best],
            "val acc. sin padding": hist["val_masked_accuracy"][best],
            "BLEU test (greedy)": bleu(test_predictions[n_units], test_refs),
            "seg / epoch": rec["seconds"] / rec["epochs_run"],
        }
    )

summary = pd.DataFrame(summary_rows).set_index("neuronas LSTM")
summary.round(3)

#### BLEU según la longitud de la oración

El encoder comprime toda la oración en el vector de contexto `(h, c)`, de tamaño fijo. Si ese cuello de botella limita al modelo, la calidad tiene que caer con la longitud de la oración, y más neuronas deberían ayudar sobre todo en las oraciones largas.

In [ ]:
length_bins = [0, 5, 10, 15, np.inf]
length_labels = ["1-5", "6-10", "11-15", "16+"]
test_bucket = pd.cut(test_en_len, bins=length_bins, labels=length_labels)

bleu_by_length = pd.DataFrame(
    {
        f"{n_units} neuronas": [
            bleu(
                [test_predictions[n_units][i] for i in np.where(test_bucket == label)[0]],
                [test_refs[i] for i in np.where(test_bucket == label)[0]],
            )
            for label in length_labels
        ]
        for n_units in runs
    },
    index=pd.Index(length_labels, name="palabras en inglés"),
)
bleu_by_length.insert(
    0, "oraciones de test", pd.Series(test_bucket).value_counts()[length_labels].values
)

bleu_by_length.drop(columns="oraciones de test").plot(marker="o", figsize=(7, 4))
plt.ylabel("BLEU")
plt.title("BLEU en test según la longitud de la oración (greedy)")
plt.grid(alpha=0.3)
plt.show()

bleu_by_length.round(2)

### Ejemplos de traducción

Seis oraciones del conjunto de test (que ningún modelo vio en entrenamiento), elegidas para cubrir distintas longitudes, traducidas por las cuatro configuraciones con decodificación greedy. Se muestran hasta dos de las traducciones de referencia.

In [ ]:
order = np.argsort(test_en_len, kind="stable")
example_positions = [order[int(q * (len(order) - 1))] for q in [0.1, 0.3, 0.5, 0.7, 0.9, 0.99]]

for k, pos in enumerate(example_positions, 1):
    i = test_idx[pos]
    print(f"[{k}] EN: {input_sentences[i]}")
    for ref in all_translations[input_sentences[i]][:2]:
        print(f"    referencia: {ref}")
    for n_units in runs:
        print(f"    {n_units:>4} neuronas: {' '.join(test_predictions[n_units][pos])}")
    print()

In [ ]:
# frases nuevas: las de clase y algunas más largas
new_sentences = [
    "My mother says hi.",
    "Where is the train station?",
    "I love learning languages.",
    "Can you help me?",
    "I don't think Tom knows where Mary lives.",
    "We should go to the beach tomorrow if the weather is good.",
    "She told me that she had never seen such a beautiful city before.",
]
new_predictions = {
    n_units: decode_batch(run, encode_english(new_sentences)) for n_units, run in runs.items()
}

for k, sentence in enumerate(new_sentences):
    print(f"EN: {sentence}")
    for n_units in runs:
        print(f"    {n_units:>4} neuronas: {' '.join(new_predictions[n_units][k])}")
    print()

### Opcional: otras estrategias de decodificación

Se comparan sobre la mejor configuración (mayor BLEU en test):

- **Greedy**: la palabra más probable en cada paso. Una mala elección temprana no se puede corregir.
- **Búsqueda por haz (beam search)**: mantiene en cada paso las `k` traducciones parciales con mayor log-probabilidad acumulada y al final elige la mejor. Como la log-probabilidad siempre suma valores negativos, favorece a las traducciones cortas; para compensar se divide el puntaje por `longitud ** alpha`.
- **Muestreo aleatorio con temperatura**: en cada paso se sortea la palabra según la distribución del modelo. La temperatura `T` la aplana (`T > 1`, más variedad y más errores) o la concentra (`T < 1`, más parecido a greedy).

In [ ]:
def beam_search(run, enc_seq, beam_width=3, alpha=0.7):
    """Traduce una oración con búsqueda por haz. Cada hipótesis: (palabras, log-prob, h, c)."""
    h, c = run["encoder"](enc_seq[None, :], training=False)
    beams = [([], 0.0, np.asarray(h)[0], np.asarray(c)[0])]
    finished = []
    for _ in range(max_out_len):
        last = np.array([[seq[-1] if seq else sos_idx] for seq, *_ in beams], dtype=np.int32)
        log_probs, h, c = decoder_step(
            run["decoder"], last, np.stack([b[2] for b in beams]), np.stack([b[3] for b in beams])
        )
        # cada hipótesis se extiende con sus beam_width palabras más probables
        candidates = []
        for b, (seq, score, _, _) in enumerate(beams):
            for idx in np.argsort(log_probs[b])[-beam_width:]:
                candidates.append((seq + [int(idx)], score + log_probs[b, idx], h[b], c[b]))
        candidates.sort(key=lambda cand: cand[1], reverse=True)

        # las que terminan en <eos> quedan completas; seguimos con las beam_width mejores del resto
        beams = []
        for cand in candidates:
            if cand[0][-1] == eos_idx:
                finished.append(cand)
            elif len(beams) < beam_width:
                beams.append(cand)
        if len(finished) >= beam_width or not beams:
            break

    best = max(finished or beams, key=lambda cand: cand[1] / len(cand[0]) ** alpha)
    return ids_to_words(best[0])


best_units = int(summary["BLEU test (greedy)"].idxmax())
best_run = runs[best_units]
print(f"Mejor configuración: {best_units} neuronas")

subset = np.arange(min(N_STRATEGY_EVAL, len(test_idx)))
subset_refs = [test_refs[i] for i in subset]
strategies = {
    "greedy": lambda: decode_batch(best_run, test_enc[subset]),
    "beam search (k=3)": lambda: [beam_search(best_run, test_enc[i], 3) for i in subset],
    "beam search (k=5)": lambda: [beam_search(best_run, test_enc[i], 5) for i in subset],
    "muestreo T=0.5": lambda: decode_batch(best_run, test_enc[subset], temperature=0.5),
    "muestreo T=1.0": lambda: decode_batch(best_run, test_enc[subset], temperature=1.0),
}
strategy_predictions = {}
strategy_rows = []
for name, decode in strategies.items():
    start = time.time()
    strategy_predictions[name] = decode()
    strategy_rows.append(
        {
            "estrategia": name,
            "BLEU": bleu(strategy_predictions[name], subset_refs),
            "palabras por traducción": np.mean([len(p) for p in strategy_predictions[name]]),
            "segundos": time.time() - start,
        }
    )
print(f"Primeras {len(subset)} oraciones de test")
strategy_summary = pd.DataFrame(strategy_rows).set_index("estrategia")
strategy_summary.round(2)

In [ ]:
# las mismas oraciones de ejemplo con cada estrategia y varias muestras por temperatura
for pos in example_positions[2:5]:
    i = test_idx[pos]
    enc = test_enc[pos : pos + 1]
    print(f"EN: {input_sentences[i]}")
    print(f"    referencia:        {all_translations[input_sentences[i]][0]}")
    print(f"    greedy:            {' '.join(decode_batch(best_run, enc)[0])}")
    for k in (3, 5):
        print(f"    beam search (k={k}): {' '.join(beam_search(best_run, enc[0], k))}")
    for temperature in (0.5, 1.0, 1.5):
        for s in range(2):
            sample = decode_batch(best_run, enc, temperature=temperature, seed=s)[0]
            print(f"    muestreo T={temperature}:    {' '.join(sample)}")
    print()

### Resumen de resultados

La celda siguiente reúne en texto los números que se usan en la interpretación.

In [ ]:
print("=== Datos ===")
print(
    f"Pares: {len(pairs):,} | train {np.sum(split == 'train'):,} | val {np.sum(split == 'val'):,} "
    f"| oraciones de test {len(test_idx):,}"
)
print(
    f"max_input_len = {max_input_len} | max_out_len = {max_out_len} "
    f"(clase: {CLASS_MAX_INPUT_LEN} / {CLASS_MAX_OUT_LEN})"
)
print(f"Pares que la clase hubiera truncado: {n_truncated} ({n_lost_sos} perdían el <sos>)")
print(f"Posiciones de padding en el decoder: {pad_share:.1%}")
print(f"Vocabulario: {nb_words:,} (inglés) / {num_words_output:,} (español)")
print(
    f"Apariciones sin vector GloVe: {glove_missing_raw:.1%} sin separar contracciones -> "
    f"{glove_missing_split:.1%} separándolas"
)

print("\n=== Resumen por configuración ===")
print(summary.round(3).to_string())
print("\n=== BLEU por longitud de la oración en inglés ===")
print(bleu_by_length.round(2).to_string())
print(f"\n=== Estrategias de decodificación ({best_units} neuronas, {len(subset)} oraciones) ===")
print(strategy_summary.round(2).to_string())

gap = summary["train acc. sin padding"] - summary["val acc. sin padding"]
print("\nBrecha train - val en accuracy sin padding (en la mejor epoch):")
print(gap.round(3).to_string())

### Interpretación

#### Qué mide cada métrica

- **Pérdida y perplejidad de validación (sin padding).** Entropía cruzada promedio por palabra real del español, con *teacher forcing*: el decoder siempre recibe la palabra correcta anterior. La perplejidad (`exp(loss)`) se puede leer como "entre cuántas palabras duda el modelo" en cada paso.
- **Accuracy sin padding.** Es la fracción de palabras reales del español que el modelo acierta. La accuracy de clase contaba también las posiciones de relleno, que son la gran mayoría del decoder y triviales de predecir: un modelo que sólo predijera padding ya obtenía un valor alto (ver la celda de padding) sin traducir nada. Se mide con *teacher forcing*, así que es optimista: al traducir, el modelo se alimenta de sus propias predicciones y un error temprano arrastra a los siguientes (*exposure bias*).
- **BLEU.** Mide la superposición de n-gramas con las traducciones de referencia sobre traducciones generadas de verdad, sin *teacher forcing*, así que es la métrica más cercana a la calidad real. Tiene límites: no reconoce sinónimos ni reordenamientos válidos que no estén entre las referencias, se calcula en minúsculas y sin puntuación, y en oraciones de 2 o 3 palabras los 4-gramas casi no existen.

#### Efecto del número de neuronas

`n_units` define a la vez el tamaño del vector de contexto (lo único que el decoder sabe de la oración en inglés), la capacidad de las dos LSTM y la dimensión del embedding del español. Los parámetros de las LSTM crecen con el cuadrado de `n_units` y los de la `Dense` final linealmente (ver la tabla de parámetros). Al comparar las configuraciones hay que mirar:

- si la **pérdida de validación y el BLEU** mejoran con más neuronas y si la mejora se achica (retornos decrecientes);
- la **brecha entre train y validación**: con más parámetros y los mismos 50.000 pares, el modelo puede memorizar el entrenamiento (sobreajuste) en lugar de generalizar;
- el **BLEU por longitud**: si las configuraciones grandes ganan sobre todo en oraciones largas, el cuello de botella del vector de contexto es una limitación real;
- el **costo**: segundos por epoch y número de epochs hasta la mejor validación.

#### Limitaciones del enfoque

- **Sin atención.** Toda la oración en inglés se comprime en `(h, c)`. Para oraciones largas esa compresión pierde información; los modelos con atención (y luego los Transformers) permiten al decoder mirar todas las posiciones del encoder en cada paso.
- **Decodificación y *exposure bias*.** El modelo se entrena viendo siempre el prefijo correcto, pero traduce a partir de sus propias predicciones. Greedy no puede corregir una mala elección temprana; beam search mitiga esto en parte.
- **Vocabulario cerrado.** Las palabras fuera de las 10.000 más frecuentes se eliminan del texto, así que el modelo no puede generar nombres propios raros ni conjugaciones poco frecuentes, y no puede copiar palabras del inglés. Una tokenización por subpalabras (BPE, WordPiece) evitaría este problema.
- **Embeddings.** GloVe sólo cubre el inglés, está congelado y deja en cero las palabras que no conoce. El embedding del español se aprende desde cero con 50.000 pares.
- **Datos.** Tatoeba tiene oraciones cortas, coloquiales y repetitivas (muchas con "Tom" y "Mary"): el modelo funciona dentro de ese dominio y no se puede esperar que traduzca textos técnicos o periodísticos. Las oraciones largas, justamente las más difíciles, son las menos frecuentes en el entrenamiento.
- **Evaluación.** El BLEU con referencias de Tatoeba sólo premia las traducciones que alguien cargó en el corpus; una traducción correcta pero distinta puntúa bajo.

### Conclusiones

> **Completar con los números de la celda "Resumen de resultados".**

1. **Datos y longitud.** Pasar de 10.000 a 50.000 pares y no truncar las secuencias obligó a corregir dos errores del notebook de clase que con poco dato no se manifestaban (el `<sos>` fuera del vocabulario y el truncado que borraba el `<sos>`). *[Completar: cuántos pares se hubieran truncado y qué `max_input_len` / `max_out_len` resultaron.]*
2. **Neuronas.** *[Completar: qué configuración tuvo el mejor BLEU y la menor pérdida de validación, cuánto mejoró respecto de las más chicas y si la mejora se achica al pasar de 256 a 512.]*
3. **Sobreajuste y costo.** *[Completar: cómo cambia la brecha train/val con el tamaño, cuántas epochs necesitó cada configuración y cuánto tardó por epoch.]*
4. **Longitud de la oración.** *[Completar: cuánto cae el BLEU de las oraciones cortas a las largas y si las configuraciones grandes reducen esa caída.]*
5. **Calidad de las traducciones.** *[Completar con los ejemplos: qué tipo de oraciones traduce bien (cortas, frecuentes) y qué errores aparecen (repeticiones, palabras omitidas, cambio de sujeto o tiempo verbal, nombres propios).]*
6. **Estrategias de decodificación.** *[Completar: si beam search mejora el BLEU frente a greedy y cómo cambian las traducciones con la temperatura del muestreo.]*